# Does soft voting beat the individual classifiers?

This notebook measures it rather than assuming it, and it is written to report
either answer. Soft voting is not a trick that cannot work: averaging genuinely
different models usually buys a little. The question is whether these five
models, on these five columns, are different enough for it to buy anything
here.

## The premise soft voting rests on

An average helps when its members make **different** mistakes. If every member
ranks the peptides almost identically, averaging reproduces that same ranking
and ROC-AUC barely moves. So the first thing measured is the rank correlation
between members, pooled and within protein. That number decides in advance how
much room an ensemble has.

## What is compared

All on the frozen combination, `PROP + LEN`, with protein-grouped five-fold
cross-validation and the same eight-candidate tuning each model receives in
phase 1.

- the five members on their own
- **soft vote**: the mean predicted probability
- **rank vote**: the mean within-protein rank, which ignores calibration.
  Probabilities from a logistic regression, a forest and a boosted ensemble are
  not on a common scale, so a plain average can be dominated by whichever
  member is most confident. Rank voting removes that objection.

## Two evaluations, because one of them is unfair

**Pooled and per-protein ROC-AUC.** The per-protein figure is the deployment
task: ranking peptides inside one protein.

**Best single, chosen honestly.** Comparing an ensemble against "the best of
five" is biased **against the ensemble**, because the best of five is a maximum
over five noisy numbers and is inflated by that alone. The nested comparison
fixes it: for each outer fold the best member is chosen using only the other
folds, then scored on the held-out one. That is what a practitioner could
actually do, and it is the comparison the ensemble deserves.

## How to read the output

If the member correlation is high, expect the ensemble to land between the best
and the mean of its members, which is not an improvement worth reporting. If it
is low and the ensemble still gains nothing, that is a stronger negative. If
the ensemble does win, the verdict prints the margin next to its spread across
proteins so the size can be judged.

## Cost

Five models times eight tuning candidates times five folds, on five columns:
about two hundred fits of cheap models. It times one fit and prints an estimate
before starting.

This notebook writes nothing into `ML/artifacts` and changes no module. It
reads B-cell only; SARS is never loaded.


In [ ]:
# ---- 1. Get the pipeline ----------------------------------------------------
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/azhar416/B-Cell-Epitopes.git"
PROJECT_DIR = Path("/kaggle/working/B-Cell-Epitopes")

if not (PROJECT_DIR / "ML" / "train.py").is_file():
    if PROJECT_DIR.exists():
        raise FileNotFoundError(
            f"{PROJECT_DIR} exists but holds no ML pipeline. Remove it, or "
            "point PROJECT_DIR at the repository root."
        )
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_DIR)])
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import itertools
import time

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics import roc_auc_score

from ML.CFG import CFG
from ML.dataset import load_labeled_dataset, make_cross_validation_splits
from ML.experiment_matrix import MODEL_TYPES, report_environment
from ML.model import build_model
from ML.pre_dataset import build_feature_frame
from ML.selection import apply_parameters, resolve_scale_pos_weight

# ---- 2. Settings ------------------------------------------------------------
FROZEN_GROUPS = ("prop", "len")   # what phase 1 froze
N_SPLITS = 5
RANDOM_SEED = 42
BIG_PROTEIN_MIN = 100             # SARS has 467 peptides; this is its size class
TUNE = True                       # the same 8-candidate rule phase 1 uses

base_cfg = CFG(DATASET_NAME="bcell")
print("Environment:")
if not report_environment():
    print()
    print("NOT the pinned image. Tolerated here: every model and every ensemble")
    print("below is measured in this same environment and on the same folds, so")
    print("a version difference moves them together.")
print()


def _find_data_dir(filename):
    bundled = PROJECT_DIR / "dataset"
    if (bundled / filename).is_file():
        return bundled
    found = sorted(Path("/kaggle/input").rglob(filename))
    if len(found) != 1:
        raise FileNotFoundError(f"Expected exactly one {filename}; found {len(found)}.")
    return found[0].parent


DATA_DIR = _find_data_dir("input_bcell.csv")

# ---- 3. Data, frozen columns, folds -----------------------------------------
frame = load_labeled_dataset(
    "bcell", DATA_DIR, max_peptide_length=base_cfg.MAX_PEPTIDE_LENGTH,
    feature_set="proposed",
)
features = build_feature_frame(
    frame, feature_set="proposed",
    use_prop_features="prop" in FROZEN_GROUPS,
    use_len_features="len" in FROZEN_GROUPS,
    use_aac_features=False, use_dpc_features=False, use_flank_features=False,
    use_peptide_features=False, use_protein_features=False,
    use_biological_features=False,
)
labels = frame["target"].to_numpy(dtype=int)
groups = frame["parent_protein_id"].to_numpy()
folds, strategy = make_cross_validation_splits(
    labels, groups, "bcell", n_splits=N_SPLITS, random_state=RANDOM_SEED,
)
fold_of = np.empty(len(labels), dtype=int)
for number, (_, test_index) in enumerate(folds, start=1):
    fold_of[test_index] = number

index_of = {}
for position, protein in enumerate(groups):
    index_of.setdefault(protein, []).append(position)
index_of = {k: np.array(v) for k, v in index_of.items()}
usable = [p for p, i in index_of.items() if labels[i].min() != labels[i].max()]
big = [p for p in usable if len(index_of[p]) >= BIG_PROTEIN_MIN]

print(f"rows {len(frame):,}   proteins {len(index_of):,}   split {strategy}")
print(f"columns {features.shape[1]} ({', '.join(features.columns)})")
print(f"proteins carrying both classes: {len(usable)}   "
      f"with >= {BIG_PROTEIN_MIN} peptides: {len(big)}")
if not big:
    # Nothing clears the size threshold, so the SARS-sized view would be
    # empty and every paired test on it undefined. Fall back rather than
    # fail, and say so, because a silent empty subset reads as a result.
    print(f"  no protein reaches {BIG_PROTEIN_MIN} peptides; the size-matched")
    print("  view falls back to all usable proteins.")
    big = list(usable)
print()


# ---- 4. Out-of-fold probabilities, one model at a time ----------------------
def oof_for(config):
    """Protein-grouped out-of-fold probabilities for one configuration."""
    out = np.full(len(labels), np.nan)
    for train_index, test_index in folds:
        model = build_model(
            model_type=config.MODEL_TYPE,
            random_state=config.RANDOM_STATE,
            random_forest_trees=config.RANDOM_FOREST_TREES,
            tree_max_depth=config.TREE_MAX_DEPTH,
            tree_min_samples_leaf=config.TREE_MIN_SAMPLES_LEAF,
            tree_min_samples_split=config.TREE_MIN_SAMPLES_SPLIT,
            tree_max_features=config.TREE_MAX_FEATURES,
            class_weight=config.CLASS_WEIGHT,
            logistic_regression_c=config.LOGISTIC_REGRESSION_C,
            logistic_regression_max_iter=config.LOGISTIC_REGRESSION_MAX_ITER,
            xgboost_estimators=config.XGBOOST_ESTIMATORS,
            xgboost_max_depth=config.XGBOOST_MAX_DEPTH,
            xgboost_learning_rate=config.XGBOOST_LEARNING_RATE,
            xgboost_subsample=config.XGBOOST_SUBSAMPLE,
            xgboost_colsample_bytree=config.XGBOOST_COLSAMPLE_BYTREE,
            xgboost_scale_pos_weight=resolve_scale_pos_weight(
                config, labels[train_index]),
            mlp_hidden_layer_sizes=config.MLP_HIDDEN_LAYER_SIZES,
            mlp_alpha=config.MLP_ALPHA,
            mlp_learning_rate_init=config.MLP_LEARNING_RATE_INIT,
            mlp_max_iter=config.MLP_MAX_ITER,
            mlp_early_stopping=config.MLP_EARLY_STOPPING,
            feature_set="proposed",
        )
        model.fit(features.iloc[train_index], labels[train_index])
        out[test_index] = model.predict_proba(features.iloc[test_index])[:, 1]
    return out


def tuned_oof(model_type):
    """The protocol's rule: score 8 candidates on pooled OOF, keep the best.

    Candidate zero is the CFG default, exactly as ML.selection builds it, so
    the tuner can never return something worse than the configured baseline.
    """
    config = CFG(DATASET_NAME="bcell", MODEL_TYPE=model_type)
    config.validate()
    overrides = (config.HYPERPARAMETER_TUNING_CANDIDATES.get(model_type, [])
                 if TUNE else [])
    best_score, best_oof, best_label = -np.inf, None, "defaults"
    for number, override in enumerate([None] + list(overrides)):
        candidate = config if override is None else apply_parameters(config, override)
        scores = oof_for(candidate)
        score = roc_auc_score(labels, scores)
        if score > best_score:
            best_score, best_oof = score, scores
            best_label = "defaults" if override is None else str(override)
        print(f"    {model_type:<20} candidate {number}/{len(overrides)}  "
              f"pooled ROC {score:.4f}")
    print(f"    {model_type:<20} -> {best_score:.4f}  {best_label}")
    return best_oof


started = time.time()
_ = oof_for(CFG(DATASET_NAME="bcell", MODEL_TYPE="logistic_regression"))
one_fit = time.time() - started
planned = len(MODEL_TYPES) * (8 if TUNE else 1)
print(f"one 5-fold pass: {one_fit:.1f}s (logistic regression)")
print(f"planned passes: {planned}  ->  rough estimate "
      f"{planned * one_fit * 3 / 60:.0f} min, trees and the MLP being slower")
print()

print("=" * 78)
print("Tuning each member on the frozen combination")
print("=" * 78)
member_oof = {}
for model_type in MODEL_TYPES:
    member_oof[model_type] = tuned_oof(model_type)
print()

SHORT = {"random_forest": "RF", "logistic_regression": "LR",
         "extra_trees": "ET", "xgboost": "XGB", "mlp": "MLP"}
members = list(MODEL_TYPES)


# ---- 5. The ensembles -------------------------------------------------------
def within_protein_ranks(scores):
    """Rank inside each protein, so calibration differences cannot dominate."""
    out = np.zeros(len(scores))
    for protein, index in index_of.items():
        out[index] = stats.rankdata(scores[index]) / len(index)
    return out


soft_vote = np.mean([member_oof[m] for m in members], axis=0)
rank_vote = np.mean([within_protein_ranks(member_oof[m]) for m in members], axis=0)
candidates = {**{SHORT[m]: member_oof[m] for m in members},
              "SOFT VOTE": soft_vote, "RANK VOTE": rank_vote}


def per_protein(scores, subset):
    return np.array([roc_auc_score(labels[index_of[p]], scores[index_of[p]])
                     for p in subset])


# ---- 6. How much do the members disagree? -----------------------------------
print("=" * 78)
print("1. Member agreement -- an average can only help if they differ")
print("=" * 78)
pairs = []
for a, b in itertools.combinations(members, 2):
    pooled = stats.spearmanr(member_oof[a], member_oof[b]).statistic
    within = np.nanmean([
        stats.spearmanr(member_oof[a][index_of[p]],
                        member_oof[b][index_of[p]]).statistic
        for p in usable
    ])
    pairs.append((SHORT[a], SHORT[b], pooled, within))
print(f"{'pair':<14}{'pooled rho':>12}{'within protein':>16}")
print("-" * 44)
for a, b, pooled, within in sorted(pairs, key=lambda r: -r[3]):
    print(f"{a + ' vs ' + b:<14}{pooled:>12.3f}{within:>16.3f}")
mean_within = float(np.mean([p[3] for p in pairs]))
print(f"{'mean':<14}{np.mean([p[2] for p in pairs]):>12.3f}{mean_within:>16.3f}")
print()
if mean_within > 0.85:
    print(f"  {mean_within:.2f} within protein: the members rank peptides almost")
    print("  identically, so an average has little left to correct.")
else:
    print(f"  {mean_within:.2f} within protein: there is real disagreement here,")
    print("  so an ensemble has room to gain. Whether it does is measured below.")
print()

# ---- 7. Pooled and per-protein scores ---------------------------------------
print("=" * 78)
print("2. Every member and every ensemble, same folds")
print("=" * 78)
table = []
for name, scores in candidates.items():
    table.append({
        "model": name,
        "pooled": roc_auc_score(labels, scores),
        f"per_protein_{len(usable)}": per_protein(scores, usable).mean(),
        f"per_protein_{len(big)}": per_protein(scores, big).mean(),
    })
scores_table = pd.DataFrame(table).set_index("model")
print(scores_table.round(4).to_string())
print()

best_member = max(members, key=lambda m: roc_auc_score(labels, member_oof[m]))
print(f"best single member on pooled ROC: {SHORT[best_member]} "
      f"({roc_auc_score(labels, member_oof[best_member]):.4f})")
print("  -- note this is a maximum over five, so it is inflated; section 4")
print("     repeats the comparison without that advantage.")
print()

# ---- 8. Paired tests per protein --------------------------------------------
print("=" * 78)
print("3. Paired per protein: ensemble minus each member")
print("=" * 78)
results = {}
for subset, label in ((usable, f"{len(usable)} proteins"),
                      (big, f"{len(big)} proteins >= {BIG_PROTEIN_MIN} peptides")):
    print(f"\n  {label}")
    print(f"  {'comparison':<26}{'delta':>9}{'wins':>10}{'t':>8}{'p':>11}")
    print("  " + "-" * 64)
    for ensemble in ("SOFT VOTE", "RANK VOTE"):
        for m in members:
            delta = per_protein(candidates[ensemble], subset) - per_protein(
                member_oof[m], subset)
            t_stat, p_value = stats.ttest_1samp(delta, 0.0)
            mark = "  *" if p_value < 0.05 else ""
            print(f"  {ensemble + ' - ' + SHORT[m]:<26}{delta.mean():>+9.4f}"
                  f"{f'{int((delta > 0).sum())}/{len(delta)}':>10}"
                  f"{t_stat:>8.2f}{p_value:>11.2e}{mark}")
            results[(label, ensemble, SHORT[m])] = (float(delta.mean()), float(p_value))
        print()

# ---- 9. The fair comparison: best member chosen without peeking -------------
print("=" * 78)
print("4. Nested: the best member picked on other folds, not on the test fold")
print("=" * 78)
print("  This is the comparison the ensemble deserves. Picking the best of five")
print("  on the same rows it is scored on inflates the single model; here the")
print("  choice is made on the four training folds only.")
print()
rows = []
for number in range(1, N_SPLITS + 1):
    test_mask = fold_of == number
    train_mask = ~test_mask
    if len(np.unique(labels[test_mask])) < 2:
        continue
    picked = max(members,
                 key=lambda m: roc_auc_score(labels[train_mask],
                                             member_oof[m][train_mask]))
    rows.append({
        "fold": number,
        "picked": SHORT[picked],
        "best_single": roc_auc_score(labels[test_mask], member_oof[picked][test_mask]),
        "soft_vote": roc_auc_score(labels[test_mask], soft_vote[test_mask]),
        "rank_vote": roc_auc_score(labels[test_mask], rank_vote[test_mask]),
    })
nested = pd.DataFrame(rows)
nested["soft - single"] = nested.soft_vote - nested.best_single
nested["rank - single"] = nested.rank_vote - nested.best_single
print(nested.round(4).to_string(index=False))
print()
print(f"  mean soft vote minus best single : {nested['soft - single'].mean():+.4f}")
print(f"  mean rank vote minus best single : {nested['rank - single'].mean():+.4f}")
print()

# ---- 10. Verdict ------------------------------------------------------------
print("=" * 78)
print("VERDICT")
print("=" * 78)
big_label = f"{len(big)} proteins >= {BIG_PROTEIN_MIN} peptides"
soft_vs_best = results[(big_label, "SOFT VOTE", SHORT[best_member])]
rank_vs_best = results[(big_label, "RANK VOTE", SHORT[best_member])]
soft_protein = per_protein(soft_vote, big)
best_protein = per_protein(member_oof[best_member], big)
spread = float((soft_protein - best_protein).std(ddof=1))

print(f"member agreement within protein      {mean_within:.3f}")
print(f"soft vote minus best member, {len(big)} prot  {soft_vs_best[0]:+.4f}  "
      f"(p={soft_vs_best[1]:.2e}, sd across proteins {spread:.4f})")
print(f"rank vote minus best member, {len(big)} prot  {rank_vs_best[0]:+.4f}  "
      f"(p={rank_vs_best[1]:.2e})")
print(f"soft vote minus nested best single    {nested['soft - single'].mean():+.4f}")
print()
gains = [soft_vs_best[0], rank_vs_best[0], float(nested["soft - single"].mean())]
if max(gains) <= 0:
    print("No ensemble beats the single models on any of the three views.")
    print("Soft voting does not improve the final result on this data.")
elif soft_vs_best[1] >= 0.05 or abs(soft_vs_best[0]) < spread:
    print("Any gain is smaller than its spread across proteins, or not")
    print("significant. That is not a result worth reporting as an improvement.")
else:
    print("Soft voting DOES gain here, significantly and beyond its spread.")
    print("Report it, with the margin and the per-protein spread above.")
print()

out = Path("/kaggle/working/softvoting_check.csv")
scores_table.to_csv(out)
print(scores_table.round(4).to_string())
print()
print(f"Saved to {out}")
